### 01 - Coleta de Dados

Este notebook realiza a primeira etapa do pipeline de dados do projeto: a coleta de informações públicas do YouTube relacionadas à repercussão do ECA Digital no contexto dos jogos eletrônicos no Brasil.

A coleta utiliza a YouTube Data API v3 e contempla três principais tipos de dados:
- metadados dos vídeos encontrados a partir de palavras-chave;
- métricas de engajamento dos vídeos, como visualizações, curtidas e quantidade de comentários;
- comentários públicos associados aos vídeos coletados.

Os dados obtidos nesta etapa serão armazenados em formato CSV na camada `data/raw`, preservando os dados em seu estado bruto para posterior tratamento, análise exploratória e processamento de linguagem natural.


In [ ]:
import sys
from pathlib import Path
import pandas as pd

Como o projeto foi organizado de forma modular, as funções auxiliares e configurações foram separadas dentro da pasta `src`.

Para garantir que o notebook consiga importar esses módulos corretamente, o diretório raiz do projeto é adicionado ao caminho de busca do Python.

In [ ]:
ROOT_DIR = Path.cwd().parent

if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

In [ ]:
from src.config import (
    YOUTUBE_API_KEY,
    DIRETORIOS_PROJETO,
    PALAVRAS_CHAVE,
    REGION_CODE,
    RELEVANCE_LANGUAGE,
    MAX_VIDEOS_POR_TERMO,
    MAX_COMENTARIOS_POR_VIDEO,
    RAW_DATA_DIR
)

from src.data_utils import (
    criar_diretorios,
    salvar_csv,
    remover_duplicados
)

from src.youtube_api import (
    criar_cliente_youtube,
    coletar_videos_por_palavras_chave,
    buscar_estatisticas_videos,
    adicionar_estatisticas_aos_videos,
    coletar_comentarios_dos_videos
)

Antes de iniciar a coleta, são criados os diretórios definidos no arquivo de configuração do projeto.

Esta etapa garante que os dados coletados sejam armazenados de forma organizada, separando os dados brutos, dados processados, arquivos finais e saídas geradas durante a análise.

In [ ]:
criar_diretorios(DIRETORIOS_PROJETO)

A conexão com a YouTube Data API v3 é realizada por meio de um cliente autenticado com a chave da API armazenada no arquivo `.env`

A chave não é inserida diretamente no código por questões de segurança e boas práticas, evitando a exposição de informações sensíveis no repositório do projeto.

In [ ]:
youtube = criar_cliente_youtube(YOUTUBE_API_KEY)

Os termos de busca foram definidos com base no tema central do projeto: a repercussão do ECA Digital no contexto dos jogos eletrônicos, da infância, da adolescência e da segurança digital.

A utilização de diferentes palavras-chave amplia a cobertura da coleta, permitindo recuperar vídeos que abordam o tema por perspectivas distintas, como regulação, proteção infantil, jogos online, influenciadores infantis e adultização no ambiente digital.

In [ ]:
PALAVRAS_CHAVE 

Nesta etapa, são coletados vídeos públicos do YouTube a partir das palavras-chave configuradas no projeto.

Para cada termo de busca, a API retorna um conjunto limitado de vídeos, priorizando resultados em português e associados ao Brasil. Cada vídeo coletado contém informações básicas, como título, descrição, canal, data de publicação, palavra-chave de origem e link de acesso.

In [ ]:
videos = coletar_videos_por_palavras_chave(
    youtube = youtube,
    palavras_chave = PALAVRAS_CHAVE,
    max_videos_por_termo = MAX_VIDEOS_POR_TERMO,
    region_code = REGION_CODE,
    relevance_language = RELEVANCE_LANGUAGE,  
)

len(videos)

df_videos = pd.DataFrame(videos)
df_videos.head()
  

Como um mesmo vídeo pode aparecer em mais de uma busca, é necessário remover registros duplicados com base no identificador único do vídeo.

Essa decisão evita que um mesmo conteúdo seja contabilizado mais de uma vez nas próximas etapas da análise.

In [ ]:
df_videos = remover_duplicados(df_videos, subset="video_id")
df_videos.shape

Após a coleta dos metadados básicos, são buscadas as estatísticas de engajamento de cada vídeo.

Essas métricas são importantes para avaliar a repercussão dos conteúdos, permitindo observar quais vídeos possuem maior alcance e interação do público.

In [ ]:
videos_id = df_videos["video_id"].tolist()

estatisticas = buscar_estatisticas_videos(youtube=youtube, video_ids=videos_id)
len(estatisticas)

In [ ]:
videos_com_estatisticas = adicionar_estatisticas_aos_videos(
    videos = df_videos.to_dict(orient="records"),
    estatisticas = estatisticas
    
)

df_videos = pd.DataFrame(videos_com_estatisticas)
df_videos.head()

Os dados dos vídeos são salvos em formato CSV na pasta `data/raw`.
Neste momento, os dados ainda são considerados brutos, pois passaram apenas por uma remoção básica de duplicidades e ainda não receberam tratamento textual, normalização ou filtragens analíticas mais avançadas.

In [ ]:
caminho_videos_raw = RAW_DATA_DIR / "videos_youtube_raw.csv"
salvar_csv(df_videos, caminho_videos_raw)

caminho_videos_raw

Além dos metadados e métricas dos vídeos, também são coletados comentários públicos associados aos vídeos encontrados.

Os comentários são especialmente relevantes para o projeto, pois representam manifestações diretas dos usuários e serão utilizados em análise de opinião, frequência de termos, sentimentos e percepção pública sobre o tema.

In [ ]:
comentarios = coletar_comentarios_dos_videos(
    youtube = youtube,
    video_ids = videos_id,
    max_comentarios_por_video = MAX_COMENTARIOS_POR_VIDEO)


len(comentarios)

In [ ]:
df_comentarios = pd.DataFrame(comentarios)
df_comentarios.head()

Assim como ocorre com os vídeos, os comentários também podem ser verificados quanto à duplicidade.

A remoção de comentários duplicados é feita com base no identificador único do comentário, garantindo maior consistência para as próximas etapas do pipeline.


In [ ]:
if not df_comentarios.empty:
    df_comentarios = remover_duplicados(df_comentarios, subset=["video_id", "comentario_id"])
    df_comentarios.shape

Os comentários coletados são salvos separadamente dos dados dos vídeos.

Essa separação facilita as próximas etapas do projeto, pois os vídeos e os comentários possuem naturezas diferentes: os vídeos serão utilizados principalmente para análise de alcance e engajamento, enquanto os comentários serão utilizados para análise textual e processamento de linguagem natural.

In [ ]:
caminho_comentarios_raw = RAW_DATA_DIR / "comentarios_youtube_raw.csv"
salvar_csv(df_comentarios, caminho_comentarios_raw)

caminho_comentarios_raw

Ao final da coleta, é realizada uma verificação inicial dos dados obtidos.

Essa etapa não corresponde ainda à análise exploratória completa, mas permite confirmar se os arquivos foram gerados corretamente, quantos registros foram coletados e quais campos estão disponíveis para as próximas fases do projeto.

In [ ]:
print(f"Total de vídeos coletados: {len(df_videos)}")
print(f"Total de comentários coletados: {len(df_comentarios)}")



In [ ]:
display(df_videos[[
    "titulo",
    "canal",
    "data_publicacao",
    "visualizacoes",
    "likes",
    "quantidade_comentarios",
    "palavra_chave"
]].head())

display(df_comentarios[[
    "video_id",
    "texto_comentario",
    "data_comentario",
    "likes_comentario"
]].head())

### Considerações finais da coleta

A primeira etapa do pipeline foi concluída com a coleta de dados públicos do YouTube relacionados ao tema do projeto.

Foram gerados dois conjuntos de dados brutos:
-  `videos_youtube_raw.csv`: contém metadados e métricas de engajamento dos vídeos coletados;
- `comentarios_youtube_raw.csv`: contém comentários públicos associados aos vídeos.

Esses arquivos serão utilizados nas próximas etapas do projeto, que envolvem tratamento dos dados, limpeza textual, análise exploratória e preparação para aplicação de técnicas de processamento de linguagem natural.